# 01 — Environment Setup, Folder Structure & Data Download

This is the first notebook of the **Chemical Industry Forecasting Suite** project.
Its only job is to prepare a clean, reproducible workspace from scratch.

Running this notebook once will:
- Verify all required libraries are installed and print their versions
- Create the full project folder structure
- Download raw data from FRED, World Bank, and EIA APIs
- Save all raw datasets as CSV files to `data/raw/`
- Generate a `config.py` file used by all subsequent notebooks

**Run this notebook before any other notebook in the project.**

## 1. Library Version Check

We verify every library needed across all notebooks is installed and accessible.
Any missing library will raise a clear error message here rather than failing
silently in a later notebook.

In [1]:
import importlib
import sys

# Dictionary of required libraries: import_name -> display_name
REQUIRED_LIBRARIES = {
    "pandas":        "pandas",
    "numpy":         "numpy",
    "matplotlib":    "matplotlib",
    "seaborn":       "seaborn",
    "plotly":        "plotly",
    "sklearn":       "scikit-learn",
    "lightgbm":      "lightgbm",
    "prophet":       "prophet",
    "darts":         "darts",
    "fredapi":       "fredapi",
    "wbdata":        "wbdata",
    "streamlit":     "streamlit",
    "optuna":        "optuna",
    "mapie":         "mapie",
    "joblib":        "joblib",
    "requests":      "requests",
    "statsmodels":   "statsmodels",
}

print(f"Python version : {sys.version}")
print("-" * 45)

all_ok = True
for import_name, display_name in REQUIRED_LIBRARIES.items():
    try:
        module = importlib.import_module(import_name)
        version = getattr(module, "__version__", "version unavailable")
        print(f"  ✓  {display_name:<20} {version}")
    except ImportError:
        print(f"  ✗  {display_name:<20} NOT INSTALLED")
        all_ok = False

print("-" * 45)
if all_ok:
    print("All libraries installed. Ready to proceed.")
else:
    print("Some libraries are missing. Install them before continuing.")

Python version : 3.13.11 | packaged by Anaconda, Inc. | (main, Dec 10 2025, 21:21:08) [Clang 20.1.8 ]
---------------------------------------------
  ✓  pandas               2.3.3
  ✓  numpy                2.3.4
  ✓  matplotlib           3.10.6
  ✓  seaborn              0.13.2
  ✓  plotly               6.3.0
  ✓  scikit-learn         1.7.2
  ✓  lightgbm             4.6.0
  ✓  prophet              1.3.0
  ✓  darts                0.43.0
  ✓  fredapi              0.5.2
  ✓  wbdata               1.1.0
  ✓  streamlit            1.51.0
  ✓  optuna               4.8.0
  ✓  mapie                1.3.0
  ✓  joblib               1.5.2
  ✓  requests             2.32.5
  ✓  statsmodels          0.14.5
---------------------------------------------
All libraries installed. Ready to proceed.


## 1b. Install Missing Libraries

Any library that failed the version check above will be installed automatically
in this cell. After installation, re-run Cell 3 to confirm all libraries are
now present and print their versions.

In [2]:
import subprocess

# Libraries that failed the import check above
MISSING = [
    name for import_name, name in REQUIRED_LIBRARIES.items()
    if importlib.util.find_spec(import_name) is None
]

if not MISSING:
    print("No missing libraries. Nothing to install.")
else:
    print(f"Installing {len(MISSING)} missing libraries...\n")
    for lib in MISSING:
        print(f"  Installing {lib}...")
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", lib, "--quiet"],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            print(f"  ✓  {lib} installed successfully")
        else:
            print(f"  ✗  {lib} failed — {result.stderr.strip()}")

    print("\nDone. Re-run Cell 3 to verify all libraries are now importable.")

No missing libraries. Nothing to install.


## 2. Project Folder Structure

We create the full directory tree that all notebooks will read from and write to.
Folders are created only if they do not already exist — re-running this notebook
is safe and will not overwrite anything.
```
chemical-forecasting/
│
├── data/
│   ├── raw/            ← original downloaded datasets, never modified
│   └── processed/      ← cleaned and feature-engineered datasets
│
├── models/             ← saved model artefacts (joblib / pickle)
│
├── outputs/
│   ├── figures/        ← saved charts (PNG) for report and dashboard
│   └── reports/        ← exported notebook reports (HTML/PDF)
│
├── notebooks/          ← all .ipynb files live here
├── config.py           ← shared constants imported by all notebooks
└── streamlit_app.py    ← dashboard application
```

In [3]:
import os

FOLDERS = [
    "data/raw",
    "data/processed",
    "models",
    "outputs/figures",
    "outputs/reports",
    "notebooks",
]

print("Creating project folder structure...\n")

for folder in FOLDERS:
    os.makedirs(folder, exist_ok=True)
    print(f"  ✓  {folder}/")

print("\nFolder structure ready.")

Creating project folder structure...

  ✓  data/raw/
  ✓  data/processed/
  ✓  models/
  ✓  outputs/figures/
  ✓  outputs/reports/
  ✓  notebooks/

Folder structure ready.


## 3. Configuration File

We write `config.py` to the project root. This file stores all shared constants:
file paths, date ranges, target variable names, and API settings.

Every subsequent notebook imports from `config.py` instead of hardcoding values.
If anything changes (e.g., the date range or a column name), you change it in
one place and it propagates everywhere.

In [4]:
config_content = '''"""
config.py — Shared constants for the Chemical Industry Forecasting Suite.
Imported by all notebooks and streamlit_app.py.
"""

# ── Date range ────────────────────────────────────────────────────────────────
START_DATE = "2000-01-01"
END_DATE   = "2024-12-31"

# ── Target variables ──────────────────────────────────────────────────────────
PRICE_TARGET  = "crude_oil_wti"      # raw material price prediction target
DEMAND_TARGET = "chemical_production_index"  # demand forecasting target

# ── Forecast horizons (weeks) ─────────────────────────────────────────────────
HORIZONS = [4, 8, 12]
DEFAULT_HORIZON = 8

# ── Train / test split ────────────────────────────────────────────────────────
TEST_CUTOFF_DATE = "2023-01-01"   # everything after this is the holdout test set

# ── Raw data file paths ───────────────────────────────────────────────────────
RAW_FRED        = "data/raw/fred_prices.csv"
RAW_WORLDBANK   = "data/raw/worldbank_commodities.csv"
RAW_EIA         = "data/raw/eia_production.csv"
RAW_M5          = "data/raw/m5_demand.csv"

# ── Processed data file paths ─────────────────────────────────────────────────
FEATURES_TRAIN  = "data/processed/features_train.csv"
FEATURES_TEST   = "data/processed/features_test.csv"

# ── Model artefact paths ──────────────────────────────────────────────────────
MODEL_PROPHET   = "models/prophet_model.pkl"
MODEL_LGBM      = "models/lgbm_model.pkl"
MODEL_TFT       = "models/tft_model"
MODEL_ENSEMBLE  = "models/ensemble_weights.pkl"

# ── Output paths ──────────────────────────────────────────────────────────────
FIGURES_DIR     = "outputs/figures/"
REPORTS_DIR     = "outputs/reports/"

# ── FRED series IDs ───────────────────────────────────────────────────────────
# Full list: https://fred.stlouisfed.org
FRED_SERIES = {
    "crude_oil_wti":              "DCOILWTICO",   # WTI crude oil spot price (weekly)
    "natural_gas_henry_hub":      "DHHNGSP",      # Henry Hub natural gas spot price
    "ppi_chemicals":              "PCU325325",    # PPI: Chemical manufacturing
    "ppi_plastics":               "PCU326326",    # PPI: Plastics & rubber products
    "industrial_production_idx":  "INDPRO",       # Industrial production index
    "capacity_utilisation":       "TCU",          # Total industry capacity utilisation
    "us_gdp":                     "GDP",          # US GDP (quarterly)
    "unemployment_rate":          "UNRATE",       # US unemployment rate
}

# ── World Bank commodity indicator codes ──────────────────────────────────────
WB_INDICATORS = {
    "crude_oil_brent":   "CRUDE_BRENT",
    "natural_gas_us":    "NGAS_US",
    "coal_aus":          "COAL_AUS",
    "fertiliser_urea":   "UREA_E_US",
    "ammonia":           "PHOSROCK",
}

# ── EIA API settings ──────────────────────────────────────────────────────────
EIA_BASE_URL    = "https://api.eia.gov/v2/"
EIA_SERIES = {
    "us_chemical_production": "ELEC.PLANT.GEN.57890-NG-99.M",
    "refinery_utilisation":   "PET.WPULEUS2.W",
}

# ── Modelling constants ───────────────────────────────────────────────────────
RANDOM_SEED         = 42
CV_N_SPLITS         = 5         # number of time-series CV folds
PREDICTION_INTERVAL = 0.90      # target coverage for prediction intervals
LGBM_N_TRIALS       = 50        # Optuna tuning trials for LightGBM
'''

with open("config.py", "w") as f:
    f.write(config_content)

print("config.py written successfully.")
print("\nPreview of key settings:")
print(f"  Start date       : 2000-01-01")
print(f"  End date         : 2024-12-31")
print(f"  Test cutoff      : 2023-01-01")
print(f"  Forecast horizons: [4, 8, 12] weeks")
print(f"  PI target        : 90%")
print(f"  CV folds         : 5")

config.py written successfully.

Preview of key settings:
  Start date       : 2000-01-01
  End date         : 2024-12-31
  Test cutoff      : 2023-01-01
  Forecast horizons: [4, 8, 12] weeks
  PI target        : 90%
  CV folds         : 5


## 4. Download Raw Data from FRED

FRED (Federal Reserve Economic Data) is our primary source for macro-economic
indicators and commodity spot prices.

We use the `fredapi` library. A free API key is required — register at:
https://fred.stlouisfed.org/docs/api/api_key.html

Each series is downloaded at the highest available frequency and saved to
`data/raw/fred_prices.csv`.

In [5]:
import pandas as pd
from fredapi import Fred
from config import FRED_SERIES, START_DATE, END_DATE, RAW_FRED

# ── API key ───────────────────────────────────────────────────────────────────
# Set your FRED API key here, or store it as an environment variable:
# export FRED_API_KEY="your_key_here"
FRED_API_KEY = os.environ.get("FRED_API_KEY", "2afd05f1bcc4dca13270f389473e6c78")

fred = Fred(api_key=FRED_API_KEY)

print(f"Downloading {len(FRED_SERIES)} FRED series from {START_DATE} to {END_DATE}...\n")

fred_data = {}
for col_name, series_id in FRED_SERIES.items():
    try:
        series = fred.get_series(
            series_id,
            observation_start=START_DATE,
            observation_end=END_DATE
        )
        series.name = col_name
        fred_data[col_name] = series
        print(f"  ✓  {col_name:<40} {len(series):>5} observations")
    except Exception as e:
        print(f"  ✗  {col_name:<40} FAILED: {e}")

# Combine into a single DataFrame, align on a weekly frequency
fred_df = pd.DataFrame(fred_data)
fred_df.index = pd.to_datetime(fred_df.index)
fred_df = fred_df.resample("W").mean()   # resample to weekly, take mean
fred_df.index.name = "date"

# Save
fred_df.to_csv(RAW_FRED)

print(f"\nSaved → {RAW_FRED}")
print(f"Shape  : {fred_df.shape[0]} rows × {fred_df.shape[1]} columns")
print(f"Period : {fred_df.index.min().date()} → {fred_df.index.max().date()}")
fred_df.head()


  ✓  crude_oil_wti                             6522 observations
  ✓  natural_gas_henry_hub                     6522 observations
  ✓  ppi_chemicals                              300 observations
  ✓  ppi_plastics                               300 observations
  ✓  industrial_production_idx                  300 observations
  ✓  capacity_utilisation                       300 observations
  ✓  us_gdp                                     100 observations
  ✓  unemployment_rate                          300 observations

Saved → data/raw/fred_prices.csv
Shape  : 1306 rows × 8 columns
Period : 2000-01-02 → 2025-01-05


,crude_oil_wti,natural_gas_henry_hub,ppi_chemicals,ppi_plastics,industrial_production_idx,capacity_utilisation,us_gdp,unemployment_rate
date,,,,,,,,
2000-01-02,NaN,NaN,153.6,123.5,91.538,82.179,10002.179,4.0
2000-01-09,24.9475,2.1750,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-16,26.2680,2.2500,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-23,29.3675,2.4575,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-30,28.3360,2.7080,NaN,NaN,NaN,NaN,NaN,NaN


### 4b. Retry Failed Series

Two series failed due to a temporary FRED server error. We retry them
individually and append them to the existing CSV.

In [6]:
import time

RETRY_SERIES = {
    "us_gdp":           "GDP",
    "unemployment_rate": "UNRATE",
}

print("Retrying failed series...\n")

# Load what we already saved
fred_df = pd.read_csv(RAW_FRED, index_col="date", parse_dates=True)

for col_name, series_id in RETRY_SERIES.items():
    for attempt in range(3):
        try:
            time.sleep(2)   # brief pause before each request
            series = fred.get_series(
                series_id,
                observation_start=START_DATE,
                observation_end=END_DATE
            )
            series.name = col_name
            # Resample to weekly and merge into existing dataframe
            series_weekly = series.resample("W").mean()
            series_weekly.index.name = "date"
            fred_df[col_name] = series_weekly
            print(f"  ✓  {col_name:<40} {len(series):>5} observations")
            break
        except Exception as e:
            print(f"  ✗  Attempt {attempt + 1}/3 failed for {col_name}: {e}")
            time.sleep(5)

# Overwrite the CSV with the complete dataframe
fred_df.to_csv(RAW_FRED)

print(f"\nUpdated → {RAW_FRED}")
print(f"Shape   : {fred_df.shape[0]} rows × {fred_df.shape[1]} columns")
fred_df.head()

Retrying failed series...

  ✓  us_gdp                                     100 observations
  ✓  unemployment_rate                          300 observations

Updated → data/raw/fred_prices.csv
Shape   : 1306 rows × 8 columns


,crude_oil_wti,natural_gas_henry_hub,ppi_chemicals,ppi_plastics,industrial_production_idx,capacity_utilisation,us_gdp,unemployment_rate
date,,,,,,,,
2000-01-02,NaN,NaN,153.6,123.5,91.538,82.179,10002.179,4.0
2000-01-09,24.9475,2.1750,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-16,26.2680,2.2500,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-23,29.3675,2.4575,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-30,28.3360,2.7080,NaN,NaN,NaN,NaN,NaN,NaN


## 5. Download Raw Data from World Bank

The World Bank provides long historical commodity price series — ideal for
training models that need to see multiple commodity cycles.

We use the `wbdata` library to pull monthly prices, then resample to weekly
frequency to match the FRED data.

In [10]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "openpyxl", "--quiet"])

CompletedProcess(args=['/opt/anaconda3/bin/python', '-m', 'pip', 'install', 'openpyxl', '--quiet'], returncode=0)

In [11]:
import requests
import io
import pandas as pd
import openpyxl
from config import START_DATE, END_DATE, RAW_WORLDBANK

# Updated URL — World Bank moved to a new document ID in 2025, xlsx format
PINK_SHEET_URL = (
    "https://thedocs.worldbank.org/en/doc/"
    "18675f1d1639c7a34d463f59263ba0a2-0050012025/related/"
    "CMO-Historical-Data-Monthly.xlsx"
)

print("Downloading World Bank Pink Sheet (Monthly xlsx)...\n")

response = requests.get(PINK_SHEET_URL, timeout=60)
response.raise_for_status()
print(f"  ✓  Downloaded ({len(response.content) / 1024:.0f} KB)")

# The xlsx has multiple sheets — we want the "Monthly Prices" sheet
xl = pd.ExcelFile(io.BytesIO(response.content))
print(f"  Sheets found: {xl.sheet_names}")

# Load the monthly prices sheet — skip the multi-row header (rows 0-3)
raw = pd.read_excel(
    io.BytesIO(response.content),
    sheet_name="Monthly Prices",
    skiprows=4,
    index_col=0
)

print(f"  Raw shape: {raw.shape}")
print(f"  First 5 index values: {list(raw.index[:5])}")
print(f"  First 5 columns: {list(raw.columns[:5])}")

# Parse date index — World Bank uses "2000M01" format
def parse_wb_date(d):
    d = str(d).strip()
    for fmt in ["%YM%m", "%Y-%m", "%b-%Y", "%m/%Y"]:
        try:
            return pd.to_datetime(d, format=fmt)
        except Exception:
            continue
    return pd.NaT

raw.index = raw.index.map(parse_wb_date)
raw = raw[raw.index.notna()].sort_index()
raw = raw.apply(pd.to_numeric, errors="coerce")

# Filter to our date range
raw = raw.loc[START_DATE:END_DATE]
print(f"\n  After date filter: {raw.shape}")
print(f"  Period: {raw.index.min().date()} → {raw.index.max().date()}")

# Select relevant commodity columns
COLS_WANTED = {
    "Crude oil, avg":        "crude_oil_avg",
    "Coal, Australia":       "coal_australia",
    "Natural gas, US":       "natural_gas_us",
    "Urea, E. Europe, bag":  "fertiliser_urea",
    "Phosphate rock":        "phosphate_rock",
}

# Exact match first, then fuzzy
selected = {}
for wanted, rename in COLS_WANTED.items():
    if wanted in raw.columns:
        selected[wanted] = rename
    else:
        # fuzzy: match on first word
        keyword = wanted.split(",")[0].lower()
        matches = [c for c in raw.columns if keyword in str(c).lower()]
        if matches:
            selected[matches[0]] = rename
            print(f"  Fuzzy match: '{wanted}' → '{matches[0]}'")
        else:
            print(f"  ✗  Not found: '{wanted}'")

wb_df = raw[list(selected.keys())].rename(columns=selected)

# Upsample monthly → weekly via linear interpolation
wb_df = wb_df.resample("W").interpolate(method="linear")
wb_df.index.name = "date"

wb_df.to_csv(RAW_WORLDBANK)

print(f"\nSaved → {RAW_WORLDBANK}")
print(f"Shape  : {wb_df.shape[0]} rows × {wb_df.shape[1]} columns")
wb_df.head()


  ✓  Downloaded (760 KB)
  Sheets found: ['AFOSHEET', 'Monthly Prices', 'Monthly Indices', 'Description', 'Index Weights']
  Raw shape: (793, 71)
  First 5 index values: [nan, '1960M01', '1960M02', '1960M03', '1960M04']
  First 5 columns: ['Crude oil, average', 'Crude oil, Brent', 'Crude oil, Dubai', 'Crude oil, WTI', 'Coal, Australian']

  After date filter: (300, 71)
  Period: 2000-01-01 → 2024-12-01
  Fuzzy match: 'Crude oil, avg' → 'Crude oil, average'
  Fuzzy match: 'Coal, Australia' → 'Coal, Australian'
  Fuzzy match: 'Urea, E. Europe, bag' → 'Urea '

Saved → data/raw/worldbank_commodities.csv
Shape  : 1301 rows × 5 columns


,crude_oil_avg,coal_australia,natural_gas_us,fertiliser_urea,phosphate_rock
date,,,,,
2000-01-02,NaN,NaN,NaN,NaN,NaN
2000-01-09,NaN,NaN,NaN,NaN,NaN
2000-01-16,NaN,NaN,NaN,NaN,NaN
2000-01-23,NaN,NaN,NaN,NaN,NaN
2000-01-30,NaN,NaN,NaN,NaN,NaN


In [12]:
wb_check = pd.read_csv(RAW_WORLDBANK, index_col="date", parse_dates=True)

print("Non-null counts per column:")
print(wb_check.notna().sum())
print(f"\nFirst non-null row:")
print(wb_check.dropna().head(1))
print(f"\nSample mid-period row (2010):")
print(wb_check.loc["2010-06-01":"2010-06-30"].dropna().head(1))

Non-null counts per column:
crude_oil_avg      1262
coal_australia     1262
natural_gas_us     1262
fertiliser_urea    1262
phosphate_rock     1262
dtype: int64

First non-null row:
            crude_oil_avg  coal_australia  natural_gas_us  fertiliser_urea  \
date                                                                         
2000-10-01      31.400167           27.15            5.02            111.1   

            phosphate_rock  
date                        
2000-10-01            44.0  

Sample mid-period row (2010):
            crude_oil_avg  coal_australia  natural_gas_us  fertiliser_urea  \
date                                                                         
2010-06-06      76.180328       87.527692         4.17899       259.397436   

            phosphate_rock  
date                        
2010-06-06      117.820513  


## 6. Download Raw Data from EIA

The U.S. Energy Information Administration (EIA) publishes weekly supply-side
data that are essential leading indicators for commodity price forecasting.
Rather than using the EIA API (which has unstable route structures across
versions), we scrape directly from the EIA DNAV browser pages using
BeautifulSoup. This approach is more reliable and requires no API key.

**Three series are collected:**

**Crude Oil Stocks** (`WCRSTUS1`)
Weekly US crude oil ending stocks excluding the Strategic Petroleum Reserve,
measured in thousand barrels. Inventory builds signal oversupply and tend to
precede price declines; inventory draws signal tightening supply and tend to
precede price increases. This is one of the most watched weekly indicators
in global energy markets.
Source: https://www.eia.gov/dnav/pet/hist/LeafHandler.ashx?n=PET&s=WCRSTUS1&f=W

**Refinery Utilisation Rate** (`WRPUPUS2`)
Weekly US refinery utilisation as a percentage of operable capacity. High
utilisation means refineries are processing crude aggressively, increasing
demand for feedstocks and supporting chemical input prices. Sudden drops
in utilisation — caused by hurricanes, maintenance, or demand shocks —
tighten feedstock supply almost immediately.
Source: https://www.eia.gov/dnav/pet/hist/LeafHandler.ashx?n=PET&s=WRPUPUS2&f=W

**Natural Gas Storage** (`NW2_EPG0_SWO_R48_BCF`)
Weekly working gas in underground storage for the Lower 48 states, measured
in billion cubic feet (Bcf). Natural gas is both a direct feedstock and an
energy source for chemical manufacturing. Storage levels follow a strong
seasonal injection/withdrawal cycle — deviations from the seasonal norm
reliably predict near-term price direction.
Source: https://www.eia.gov/dnav/ng/hist/nw2_epg0_swo_r48_bcfw.htm

**Why scrape instead of using the API?**
The EIA API v2 restructured its route hierarchy and the v1 series endpoint
is being deprecated. Direct HTML scraping of the DNAV pages is stable,
reproducible, and has been the fallback used by energy data practitioners
for years. The data table structure on these pages has not changed since
the early 2000s.

In [29]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from config import RAW_EIA, START_DATE, END_DATE

EIA_PAGES = {
    "crude_oil_stocks": {
        "url":   "https://www.eia.gov/dnav/pet/hist/LeafHandler.ashx?n=PET&s=WCRSTUS1&f=W",
        "label": "Weekly US crude oil ending stocks excl. SPR (thousand barrels)"
    },
    "refinery_utilisation": {
        "url":   "https://www.eia.gov/dnav/pet/hist/LeafHandler.ashx?n=PET&s=WRPUPUS2&f=W",
        "label": "Weekly US refinery utilisation rate (%)"
    },
    "natural_gas_storage": {
        "url":   "https://www.eia.gov/dnav/ng/hist/nw2_epg0_swo_r48_bcfw.htm",
        "label": "Weekly US natural gas in underground storage (Bcf)"
    },
}

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

def parse_eia_dnav_table(html_text, col_name):
    """
    Parse EIA DNAV weekly table directly from HTML using BeautifulSoup.

    Confirmed structure:
      Header row 0 : 'Year-Month'(rowspan=2) | 'Week 1'(colspan=2) | ... | 'Week 5'(colspan=2)
      Header row 1 : 'End Date' | 'Value' | 'End Date' | 'Value' | ... (10 cells, no Year-Month)
      Data rows    : 11 cells — Year-Month + 5 × (End Date, Value)

    Fix: prepend 'Year-Month' to the 10 subheader cells → 11 column names total.
    """
    soup = BeautifulSoup(html_text, "lxml")

    # Find the largest table — confirmed as the data table from diagnostic
    all_tables = soup.find_all("table")
    table = max(all_tables, key=lambda t: len(t.find_all("tr")))
    rows  = table.find_all("tr")

    # ── Build column names ────────────────────────────────────────────────────
    # Header row 1 has 10 <th>: End Date / Value × 5 weeks
    # Year-Month has rowspan=2 so it is absent from row 1 — prepend it manually
    header_rows  = [r for r in rows if r.find("th")]
    subheader_th = header_rows[-1].find_all("th")   # row 1: 10 cells
    col_names    = ["Year-Month"] + [th.get_text(strip=True) for th in subheader_th]
    # Result: ['Year-Month', 'End Date', 'Value', 'End Date', 'Value', ... × 5]

    # ── Parse data rows ───────────────────────────────────────────────────────
    data_rows = []
    for row in rows:
        tds = row.find_all("td")
        if not tds:
            continue
        cells = [td.get_text(strip=True) for td in tds]
        # Pad or trim to exactly 11 columns
        cells = cells[:len(col_names)]
        cells += [""] * (len(col_names) - len(cells))
        data_rows.append(cells)

    df_raw = pd.DataFrame(data_rows, columns=col_names)

    # ── Filter to valid Year-Month rows ───────────────────────────────────────
    df_raw = df_raw[
        df_raw["Year-Month"].str.match(r"^\d{4}-\w+$", na=False)
    ].copy()

    # Extract year (e.g. '1982-Aug' → 1982)
    df_raw["year"] = df_raw["Year-Month"].str[:4].astype(int)

    # ── Pair End Date + Value columns by position ─────────────────────────────
    # Positions: (1,2), (3,4), (5,6), (7,8), (9,10) → 5 week pairs
    records = []
    for i in range(1, len(col_names) - 1, 2):
        temp = pd.DataFrame({
            "year":     df_raw["year"].values,
            "end_date": df_raw.iloc[:, i].values,
            "value":    df_raw.iloc[:, i + 1].values,
        })
        # Keep only rows with both end_date and value populated
        temp = temp[
            (temp["end_date"].astype(str).str.strip() != "") &
            (temp["value"].astype(str).str.strip()    != "")
        ]
        records.append(temp)

    long_df = pd.concat(records, ignore_index=True)

    # ── Reconstruct full date: 'MM/DD' + year → datetime ─────────────────────
    long_df["date_str"] = (
        long_df["end_date"].astype(str).str.strip() +
        "/" +
        long_df["year"].astype(str)
    )
    long_df["date"] = pd.to_datetime(
        long_df["date_str"], format="%m/%d/%Y", errors="coerce"
    )

    # Remove thousands separator commas and convert to numeric
    long_df["value"] = (
        long_df["value"]
        .astype(str)
        .str.replace(",", "", regex=False)
    )
    long_df["value"] = pd.to_numeric(long_df["value"], errors="coerce")
    long_df = long_df.dropna(subset=["date", "value"])

    result = (
        long_df
        .set_index("date")[["value"]]
        .rename(columns={"value": col_name})
        .sort_index()
        .loc[START_DATE:END_DATE]
    )
    return result


print("Scraping EIA DNAV pages...\n")

eia_frames = {}

for col_name, info in EIA_PAGES.items():
    print(f"  {col_name}")
    try:
        response = requests.get(info["url"], headers=HEADERS, timeout=30)
        response.raise_for_status()

        df_temp = parse_eia_dnav_table(response.text, col_name)
        eia_frames[col_name] = df_temp
        print(f"  ✓  {len(df_temp):>5} obs  |  {info['label']}\n")

    except requests.exceptions.ConnectionError:
        print(f"  ✗  Connection error\n")
    except requests.exceptions.HTTPError as e:
        print(f"  ✗  HTTP error: {e}\n")
    except Exception as e:
        import traceback
        print(f"  ✗  FAILED: {type(e).__name__}: {e}")
        traceback.print_exc()
        print()

# ── Combine, resample, save ───────────────────────────────────────────────────
if eia_frames:
    eia_df = pd.concat(eia_frames.values(), axis=1).sort_index()
    eia_df = eia_df.resample("W").mean()
    eia_df.index.name = "date"
    eia_df.to_csv(RAW_EIA)

    print(f"Saved → {RAW_EIA}")
    print(f"Shape  : {eia_df.shape[0]} rows × {eia_df.shape[1]} columns")
    print(f"Period : {eia_df.index.min().date()} → {eia_df.index.max().date()}")
    eia_df.head()

else:
    print("✗  No EIA data scraped.")
    for name, info in EIA_PAGES.items():
        print(f"   {name}: {info['url']}")

Scraping EIA DNAV pages...

  crude_oil_stocks
  ✓   1304 obs  |  Weekly US crude oil ending stocks excl. SPR (thousand barrels)

  refinery_utilisation
  ✓   1304 obs  |  Weekly US refinery utilisation rate (%)

  natural_gas_storage
  ✓    783 obs  |  Weekly US natural gas in underground storage (Bcf)

Saved → data/raw/eia_production.csv
Shape  : 1304 rows × 3 columns
Period : 2000-01-09 → 2024-12-29


## 7. Download M5 Demand Data (Kaggle)

The M5 Forecasting competition dataset is the benchmark dataset for the
demand forecasting use case in this project. It contains 5+ years of daily
unit sales for approximately 30,000 Walmart products across 10 stores in
3 US states (California, Texas, and Wisconsin), running from January 2011
to May 2016.

**Why M5 for a chemical industry project?**
Chemical companies do not publish their demand data publicly. M5 serves as
a high-quality, well-studied open proxy that exhibits the same forecasting
challenges relevant to chemical product demand: strong weekly and annual
seasonality, promotional spikes, product hierarchy effects, and intermittent
demand at the SKU level. It is also a competition benchmark with hundreds
of published model results, making our own results directly comparable to
the state of the art.

**What we use from the dataset:**
- `sales_train_evaluation.csv` — daily unit sales for all 30,490 products
- `calendar.csv` — maps day IDs (d_1 through d_1941) to actual dates

Daily sales across all products and stores are aggregated to a single
**weekly total demand series** (278 weeks, January 2011 to May 2016).
This gives a clean univariate series with a realistic demand signal —
weekly totals range from ~64,000 to ~327,000 units — suitable for
benchmarking all forecasting models in notebook 04.

**How to download:**
The Kaggle CLI has authentication issues with the new KGAT token format
in version 2.0. The most reliable approach is a direct browser download:

1. Go to https://www.kaggle.com/competitions/m5-forecasting-accuracy/data
2. Sign in and accept the competition rules if prompted
3. Click **Download All** — saves `m5-forecasting-accuracy.zip`
4. Move the zip file into the project folder:
   `mv ~/Downloads/m5-forecasting-accuracy.zip data/raw/m5_temp/`

The cell below handles unzipping, reshaping, and aggregation automatically
once the zip file is in place.
```

In [31]:
import subprocess
import pandas as pd
import os
from config import RAW_M5

M5_DIR = "data/raw/m5_temp"
os.makedirs(M5_DIR, exist_ok=True)

# ── Step 1: Unzip ─────────────────────────────────────────────────────────────
print("Step 1/4  Unzipping M5 data...\n")

zip_path = f"{M5_DIR}/m5-forecasting-accuracy.zip"

if not os.path.exists(zip_path):
    raise FileNotFoundError(
        f"Zip file not found at {zip_path}\n"
        f"Please move the downloaded zip there:\n"
        f"  mv ~/Downloads/m5-forecasting-accuracy.zip {zip_path}"
    )

subprocess.run(
    ["unzip", "-o", "-q", zip_path, "-d", M5_DIR],
    check=True
)
csv_files = [f for f in os.listdir(M5_DIR) if f.endswith(".csv")]
print(f"  ✓  Unzipped")
print(f"  CSV files found: {csv_files}\n")

# ── Step 2: Load sales and calendar ──────────────────────────────────────────
print("Step 2/4  Loading sales and calendar data...")

sales_df = pd.read_csv(f"{M5_DIR}/sales_train_evaluation.csv")
calendar = pd.read_csv(f"{M5_DIR}/calendar.csv")[["d", "date"]]

print(f"  ✓  Sales shape    : {sales_df.shape}")
print(f"  ✓  Calendar shape : {calendar.shape}\n")

# ── Step 3: Melt wide → long ──────────────────────────────────────────────────
print("Step 3/4  Reshaping wide → long format...")

id_cols  = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
day_cols = [c for c in sales_df.columns if c.startswith("d_")]

print(f"  Day columns : {len(day_cols)}  ({day_cols[0]} → {day_cols[-1]})")

sales_long = sales_df.melt(
    id_vars=id_cols,
    value_vars=day_cols,
    var_name="d",
    value_name="sales"
)
sales_long = sales_long.merge(calendar, on="d", how="left")
sales_long["date"] = pd.to_datetime(sales_long["date"])

print(f"  ✓  Long format rows : {len(sales_long):,}\n")

# ── Step 4: Aggregate to weekly total demand ──────────────────────────────────
print("Step 4/4  Aggregating to weekly total demand...")

weekly = (
    sales_long
    .groupby("date")["sales"]
    .sum()
    .rename("total_sales")
    .resample("W")
    .sum()
)
weekly.index.name = "date"
weekly.to_csv(RAW_M5)

print(f"  ✓  Aggregated to weekly")
print(f"\nSaved → {RAW_M5}")
print(f"Shape  : {weekly.shape[0]} rows × 1 column")
print(f"Period : {weekly.index.min().date()} → {weekly.index.max().date()}")
print(f"Sales range : {weekly.min():,.0f} → {weekly.max():,.0f} units/week")
weekly.head(8)

Step 1/4  Unzipping M5 data...

  ✓  Unzipped
  CSV files found: ['sales_train_evaluation.csv', 'calendar.csv', 'sell_prices.csv', 'sales_train_validation.csv', 'sample_submission.csv']

Step 2/4  Loading sales and calendar data...
  ✓  Sales shape    : (30490, 1947)
  ✓  Calendar shape : (1969, 2)

Step 3/4  Reshaping wide → long format...
  Day columns : 1941  (d_1 → d_1941)
  ✓  Long format rows : 59,181,090

Step 4/4  Aggregating to weekly total demand...
  ✓  Aggregated to weekly

Saved → data/raw/m5_demand.csv
Shape  : 278 rows × 1 column
Period : 2011-01-30 → 2016-05-22
Sales range : 64,380 → 326,832 units/week


date
2011-01-30     64380
2011-02-06    196230
2011-02-13    193715
2011-02-20    172328
2011-02-27    166645
2011-03-06    179283
2011-03-13    181714
2011-03-20    177597
Freq: W-SUN, Name: total_sales, dtype: int64

## 8. Verify All Raw Data Files

Final gate check — confirm all four raw datasets exist, have the expected
shape, and cover the correct date range before moving to EDA.

If any file shows MISSING, re-run the relevant download cell above.

In [32]:
import pandas as pd
import os
from config import RAW_FRED, RAW_WORLDBANK, RAW_EIA, RAW_M5

FILES = {
    "FRED macro & prices":     RAW_FRED,
    "World Bank commodities":  RAW_WORLDBANK,
    "EIA supply & stocks":     RAW_EIA,
    "M5 demand (Walmart)":     RAW_M5,
}

EXPECTED_COLS = {
    RAW_FRED:       8,
    RAW_WORLDBANK:  5,
    RAW_EIA:        3,
    RAW_M5:         1,
}

print("=" * 72)
print(f"  {'Dataset':<28} {'Rows':>6}  {'Cols':>5}  {'Start':<12} {'End':<12}  Status")
print("=" * 72)

all_ok = True
for name, path in FILES.items():
    if not os.path.exists(path):
        print(f"  ✗  {name:<28} {'—':>6}  {'—':>5}  {'—':<12} {'—':<12}  MISSING")
        all_ok = False
        continue

    df       = pd.read_csv(path, index_col=0, parse_dates=True)
    rows, cols = df.shape
    start    = df.index.min().date()
    end      = df.index.max().date()
    size_kb  = os.path.getsize(path) / 1024
    expected = EXPECTED_COLS.get(path, "?")
    col_ok   = "✓" if cols == expected else f"! expected {expected}"

    print(
        f"  ✓  {name:<28} {rows:>6}  {cols:>5}  "
        f"{str(start):<12} {str(end):<12}  "
        f"{size_kb:.0f} KB  cols {col_ok}"
    )

print("=" * 72)
if all_ok:
    print("\n  All raw data files present. Proceed to 02_eda.ipynb.")
else:
    print("\n  Some files are missing. Re-run the relevant download cells.")

  Dataset                        Rows   Cols  Start        End           Status
  ✓  FRED macro & prices            1306      8  2000-01-02   2025-01-05    53 KB  cols ✓
  ✓  World Bank commodities         1301      5  2000-01-02   2024-12-01    114 KB  cols ✓
  ✓  EIA supply & stocks            1304      3  2000-01-09   2024-12-29    42 KB  cols ✓
  ✓  M5 demand (Walmart)             278      1  2011-01-30   2016-05-22    5 KB  cols ✓

  All raw data files present. Proceed to 02_eda.ipynb.


## Notebook 01 Complete

All workspace components are ready:

| Component               | Details                                           |
|-------------------------|---------------------------------------------------|
| Library check           | All packages verified                             |
| Folder structure        | `data/`, `models/`, `outputs/` created            |
| `config.py`             | Shared constants for all notebooks                |
| FRED data               | 8 macro & price series, 2000–2025, weekly         |
| World Bank data         | 5 commodity price series, 2000–2024, weekly       |
| EIA data                | 3 supply & inventory series, 2000–2024, weekly    |
| M5 demand data          | Weekly aggregated Walmart demand, 2011–2016       |

**Next → `02_eda.ipynb`**

The EDA notebook will load all four raw files, merge them into a single
analytical dataset, and systematically explore patterns, seasonality,
stationarity, and cross-variable correlations that will directly guide
feature engineering decisions in notebook 03.